# ❤️ Notebook 9b — Pulse Physiology Bridge
## Build-order Step 6 | CVD Digital Twin Project | CAD_DT_Final

---

## Purpose
Takes the same raw patient dict Step 5 (NB9a) scores, maps the Pulse-compatible subset of fields into a Pulse patient, runs baseline stabilization, applies a "what-if" intervention (exercise, smoking cessation), advances the simulation, and reports before/after HR, SBP, DBP, MAP, CO.

---

## ⚠️ READ THIS FIRST — verification status (2026-08-03)
**I cannot execute or test this notebook.** Pulse's compiled binding (`PyPulse.cp313-win_amd64.pyd`) is a Windows-specific compiled binary — it doesn't exist in my environment, and there's no way for me to install or run it here. Everything below is either **confirmed** from your teammate's actual working `test_pulse.py`, or **my best engineering judgment** based on Pulse's general architecture (I have some background familiarity with the Pulse Physiology Engine project from training, but not this specific installed version's exact current API) — clearly marked either way, cell by cell.

| Piece | Status |
|---|---|
| Engine creation, data requests, `pull_data()`, `advance_time_s()`, `clear()` | ✅ **Confirmed** — copied directly from `test_pulse.py` |
| Loading a fixed pre-built patient state (`serialize_from_file`) | ✅ **Confirmed** — but this loads a *canned* patient, not one built from your raw dict's age/sex/height/weight/BP |
| **Building a custom patient from demographics** (age/sex/height/weight/BP) | ⚠️ **Unverified — this is the biggest open risk.** `test_pulse.py` only demonstrates loading a fixed file; it never shows configuring a patient from scratch. I've written this against Pulse's typical `SEPatientConfiguration`-style pattern, but the exact class names, setter methods, and whether `serialize_from_file` even has a config-object counterpart in *this* installed version are not confirmed. **`HowTo_EngineUse.py` almost certainly shows the real version of this — check it against Section 3 below before trusting anything past that point.** |
| **Applying an intervention** (exercise, smoking cessation) | ⚠️ **Unverified.** No action-application call appears anywhere in `test_pulse.py`. Written against Pulse's typical action-object + `process_action()` pattern — **`HowTo_Exercise.py` or `HowTo_CardiovascularModification.py` will show the real call; check Section 5 against whichever you have.** |
| `results[0]` = SimTime, `results[1..4]` = your 4 requests in order | ⚠️ **Inferred, not confirmed.** `test_pulse.py` indexes `results[1]` through `results[4]` for 4 declared requests and never prints `results[0]` — strongly suggesting index 0 is simulation time, but this is pattern-matching on one example, not documentation. Confirm before trusting index alignment if you add/reorder data requests.
| `MeanArterialPressure` as a valid physiology request name | ⚠️ **Not in `test_pulse.py`** (which only requests HR/SBP/DBP/CO). It's an extremely standard Pulse output name, so I'm confident enough to include it, but it's not from your confirmed example — if Section 2's data request list throws an error, this is the first thing to check.

**Bottom line: run this in your actual Pulse environment before trusting any output. Every `⚠️` above is a specific place to look for an error, and the print statements throughout are written to tell you exactly which stage failed if something breaks, rather than a bare traceback.**


---
# Section 1 — Setup & Imports

## What is being done
Imports confirmed from `test_pulse.py`, plus a few additional imports needed for patient configuration and actions (Section 3/5) that don't appear in `test_pulse.py` but follow Pulse's standard `pulse.cdm.*` module layout — **these import paths themselves are unverified**; if any of them fail, that's the first sign this Pulse version organizes things differently.

**Update `PULSE_INSTALL_PATH` for your own machine** — your teammate's path (`D:\pulse-engine\build\install\python`) is Windows-specific and won't match your own setup, per your note that your local install looks different.


In [ ]:
import sys
import os

# ── Update this for your machine — confirmed pattern from test_pulse.py, ────────
# but the exact path is obviously per-machine, not something to hardcode blindly.
PULSE_INSTALL_PATH = r"D:\pulse-engine\build\install\python"
sys.path.insert(0, PULSE_INSTALL_PATH)

# ── Confirmed imports (from test_pulse.py) ───────────────────────────────────────
from pulse.engine.PulseEngine import PulseEngine
from pulse.cdm.engine import SEDataRequestManager, SEDataRequest
from pulse.cdm.scalars import FrequencyUnit, PressureUnit, VolumePerTimeUnit

# ── Unverified imports (needed for Sections 3/5, not present in test_pulse.py) ───
# If any of these fail, check HowTo_EngineUse.py / HowTo_Exercise.py for the
# actual module paths this Pulse version uses — do not assume these are correct.
_patient_config_available = True
_action_imports_available = True
try:
    from pulse.cdm.patient import SEPatientConfiguration, SEPatient
    from pulse.cdm.scalars import LengthUnit, MassUnit
except ImportError as e:
    print(f"⚠️  Patient-configuration imports failed: {e}")
    print("    Section 3 (custom patient from demographics) will not work until "
          "this is fixed — check HowTo_EngineUse.py for the real import paths.")
    _patient_config_available = False

try:
    from pulse.cdm.patient_actions import SEExercise, SESmokingCessation
except ImportError as e:
    print(f"⚠️  Action imports failed: {e}")
    print("    Section 5 (apply intervention) will not work until this is fixed — "
          "check HowTo_Exercise.py / HowTo_CardiovascularModification.py for the "
          "real import paths and action class names.")
    _action_imports_available = False

print("\n✅ Section 1 complete")
print(f"   Patient-config imports available : {_patient_config_available}")
print(f"   Action imports available         : {_action_imports_available}")


---
# Section 2 — Data Requests

## What is being done
Declares which physiology outputs to pull each time `pull_data()` is called. The first four (HeartRate, SystolicArterialPressure, DiastolicArterialPressure, CardiacOutput) are **confirmed exactly** from `test_pulse.py`, including their unit types. `MeanArterialPressure` is added for the spec's MAP requirement — **this specific line is unverified** (see header); if it throws, comment it out and compute MAP manually as `(SBP + 2*DBP) / 3` instead (a standard clinical approximation, not Pulse's own value, but a reasonable fallback).


In [ ]:
data_requests = [
    SEDataRequest.create_physiology_request("HeartRate", unit=FrequencyUnit.Per_min),
    SEDataRequest.create_physiology_request("SystolicArterialPressure", unit=PressureUnit.mmHg),
    SEDataRequest.create_physiology_request("DiastolicArterialPressure", unit=PressureUnit.mmHg),
    SEDataRequest.create_physiology_request("CardiacOutput", unit=VolumePerTimeUnit.L_Per_min),
    # ── UNVERIFIED (added 2026-08-03, not in test_pulse.py) ──────────────────────
    # If this throws a KeyError/name-not-found style error, MAP is either named
    # differently in this Pulse version or needs a different request constructor.
    # Fallback: remove this line and compute MAP = (SBP + 2*DBP) / 3 from the
    # confirmed SBP/DBP values instead (see extract_state() in Section 4).
    SEDataRequest.create_physiology_request("MeanArterialPressure", unit=PressureUnit.mmHg),
]

# Index map — CONFIRMED for indices 0-3 pattern (test_pulse.py indexes results[1]
# through results[4] for its 4 requests, implying results[0] is SimTime). Index 4
# (MAP) is an inferred extension of that same pattern, not independently confirmed.
RESULT_INDEX = {
    'sim_time': 0,
    'heart_rate': 1,
    'systolic_bp': 2,
    'diastolic_bp': 3,
    'cardiac_output': 4,
    'map': 5,   # UNVERIFIED index — depends on MeanArterialPressure request above working
}

data_manager = SEDataRequestManager(data_requests)
print(f"✅ {len(data_requests)} data requests declared")


---
# Section 3 — Build a Pulse Patient From Raw Demographics ⚠️ HIGHEST RISK SECTION

## What is being done
`build_pulse_patient()` maps the Pulse-compatible subset of a raw patient dict (age, sex, height, weight, systolic/diastolic BP) into a Pulse patient configuration, then initializes the engine with it — **as an alternative to** `test_pulse.py`'s `serialize_from_file("./states/StandardMale@0s.json", ...)`, which only loads a fixed canned patient and has no way to take your patient's actual values.

## Why this is flagged so heavily
None of this is demonstrated in `test_pulse.py`. It's written against Pulse's typical `SEPatientConfiguration` pattern (a patient object with unit-aware setters, passed to an engine-initialization call), based on general familiarity with how the Pulse project is usually structured — **not confirmed against this specific installed version.** `HowTo_EngineUse.py` is exactly the file that would show the real version of this. Two things are likely to differ from what's below: the exact setter method names (e.g. `set_age().set_value()` vs. some other pattern), and whether initialization happens via a method on `pulse` itself or a separate builder call.

## Smoking/activity: not mapped here
The spec asks for smoking and activity/exercise to be mapped into "Pulse's patient file / initial conditions," but Pulse's patient object typically represents smoking history as a *conditioning* factor (e.g. chronic smoking history affecting baseline lung/cardiovascular state) rather than a simple settable field, and I don't know this version's exact mechanism for that. Smoking is instead handled as an **intervention** in Section 5 (smoking-cessation action), which is a demonstrated Pulse pattern-type even though the exact class isn't confirmed. Baseline activity level has no attempted mapping at all here — flagging that gap explicitly rather than guessing at a setter that may not exist.


In [ ]:
def build_pulse_patient(raw_patient: dict):
    """
    Maps Pulse-compatible fields from a raw patient dict into a Pulse patient
    configuration. UNVERIFIED — see Section 3 markdown. Uses the SAME raw field
    names as NB9a's build_lifestyle_features()/build_clinical_features() so the
    same raw dict can feed both the ML scoring path and this Pulse path.

    Raw fields used: age, sex/gender, height (cm), weight (kg),
    ap_hi/systolic_bp, ap_lo/diastolic_bp.
    """
    if not _patient_config_available:
        raise RuntimeError(
            "Patient-configuration imports failed in Section 1 — cannot build a "
            "custom patient. Check HowTo_EngineUse.py for the correct import paths "
            "before using this function."
        )

    age = raw_patient.get('age')
    sex_raw = raw_patient.get('sex', raw_patient.get('gender'))
    height = raw_patient.get('height', raw_patient.get('height_cm'))
    weight = raw_patient.get('weight', raw_patient.get('weight_kg'))
    sbp = raw_patient.get('ap_hi', raw_patient.get('systolic_bp', raw_patient.get('resting bp s')))
    dbp = raw_patient.get('ap_lo', raw_patient.get('diastolic_bp'))

    missing = [n for n, v in [('age', age), ('sex', sex_raw), ('height', height),
                              ('weight', weight)] if v is None]
    if missing:
        raise ValueError(f'Missing Pulse-required field(s): {missing}')

    # ── UNVERIFIED from here down — general Pulse SEPatientConfiguration pattern ──
    patient = SEPatient()
    patient.get_age().set_value(age, 'yr')
    sex_str = 'Male' if sex_raw in ('m', 1, '1') else 'Female'
    patient.set_sex(sex_str)   # UNVERIFIED: exact setter/enum name not confirmed
    patient.get_height().set_value(height, LengthUnit.cm)
    patient.get_weight().set_value(weight, MassUnit.kg)
    if sbp is not None:
        patient.get_systolic_arterial_pressure_baseline().set_value(sbp, PressureUnit.mmHg)
    if dbp is not None:
        patient.get_diastolic_arterial_pressure_baseline().set_value(dbp, PressureUnit.mmHg)

    patient_configuration = SEPatientConfiguration()
    patient_configuration.set_patient(patient)
    return patient_configuration


def initialize_engine_from_patient(pulse_engine, patient_configuration, data_mgr):
    """UNVERIFIED. test_pulse.py only demonstrates serialize_from_file() with a
    fixed state file — this attempts the equivalent for a built-from-scratch
    patient. If pulse_engine has no such method, check HowTo_EngineUse.py for the
    real initialization call — this may need to be a differently-named method,
    or may require serializing the configuration to a temp file first.
    """
    if hasattr(pulse_engine, 'initialize_engine'):
        return pulse_engine.initialize_engine(patient_configuration, data_mgr)
    raise AttributeError(
        "PulseEngine has no 'initialize_engine' method in this installed version. "
        "Check HowTo_EngineUse.py for the correct method name to initialize from a "
        "patient configuration object rather than a state file."
    )


print("✅ build_pulse_patient() / initialize_engine_from_patient() defined (UNVERIFIED)")


---
# Section 4 — Baseline Stabilization & State Extraction

## What is being done
`extract_state()` wraps `pull_data()` (confirmed) into a clean dict. Baseline capture immediately after loading/initializing — confirmed pattern from `test_pulse.py` (it reads state right after `serialize_from_file` succeeds, with no explicit separate "stabilize" call). If a distinct stabilization step exists in this Pulse version (common in physiology engines — advancing a short fixed period to let the model settle before treating values as "baseline"), `test_pulse.py` doesn't show one being called explicitly, so none is added here beyond what loading already does.


In [ ]:
def extract_state(pulse_engine) -> dict:
    """Confirmed pattern (pull_data + index access) from test_pulse.py.
    MAP falls back to a computed approximation if the MAP data request
    (Section 2, unverified) isn't available in results.
    """
    results = pulse_engine.pull_data()
    state = {
        'heart_rate_bpm': results[RESULT_INDEX['heart_rate']],
        'systolic_bp_mmHg': results[RESULT_INDEX['systolic_bp']],
        'diastolic_bp_mmHg': results[RESULT_INDEX['diastolic_bp']],
        'cardiac_output_L_per_min': results[RESULT_INDEX['cardiac_output']],
    }
    try:
        state['map_mmHg'] = results[RESULT_INDEX['map']]
    except IndexError:
        # Fallback: MAP request (Section 2) unavailable — compute the standard
        # clinical approximation instead of Pulse's own simulated value.
        state['map_mmHg'] = (state['systolic_bp_mmHg'] + 2 * state['diastolic_bp_mmHg']) / 3
        state['map_source'] = 'computed_approximation'
    else:
        state['map_source'] = 'pulse_direct'
    return state


print("✅ extract_state() defined")


---
# Section 5 — Apply "What-If" Intervention ⚠️ UNVERIFIED

## What is being done
`apply_intervention()` applies an exercise or smoking-cessation action to the engine before advancing. **No action-application call appears anywhere in `test_pulse.py`** — this is written against Pulse's typical pattern (an action object with settable intensity/duration, passed to a `process_action()` call), which I have some general familiarity with from the Pulse project's usual structure, but nothing here is confirmed for this installed version.

**`HowTo_Exercise.py` and/or `HowTo_CardiovascularModification.py` will show the real call — check the action class name, its constructor/setter methods, and how it's actually submitted to the engine (`process_action` is a guess at the method name, not confirmed) before trusting this section.**


In [ ]:
def apply_intervention(pulse_engine, intervention: str, **kwargs):
    """
    UNVERIFIED — see Section 5 markdown. intervention: 'exercise' or 'smoking_cessation'.
    kwargs for 'exercise': intensity (0.0-1.0, default 0.5)
    """
    if not _action_imports_available:
        raise RuntimeError(
            "Action imports failed in Section 1 — cannot apply an intervention. "
            "Check HowTo_Exercise.py / HowTo_CardiovascularModification.py for the "
            "correct import paths and action class names before using this function."
        )

    if not hasattr(pulse_engine, 'process_action'):
        raise AttributeError(
            "PulseEngine has no 'process_action' method in this installed version. "
            "Check the HowTo action examples for the correct method name to submit "
            "an action to the engine."
        )

    if intervention == 'exercise':
        intensity = kwargs.get('intensity', 0.5)
        action = SEExercise()
        action.get_intensity().set_value(intensity)   # UNVERIFIED setter name
        pulse_engine.process_action(action)
    elif intervention == 'smoking_cessation':
        action = SESmokingCessation()
        pulse_engine.process_action(action)
    else:
        raise ValueError(f"Unknown intervention: {intervention!r} (expected 'exercise' or 'smoking_cessation')")

    print(f"  ⚠️  Applied '{intervention}' (unverified action call) — check output sanity")


print("✅ apply_intervention() defined (UNVERIFIED)")


---
# Section 6 — Combined Bridge Function

## What is being done
`run_pulse_bridge()` combines Sections 3-5 into the single before/after flow the spec asks for: build patient → initialize → capture baseline → apply intervention → advance → capture new state. Returns a clean dict shaped to compose with NB9a's output in the eventual Step 7 `simulate_patient()`.


In [ ]:
def run_pulse_bridge(raw_patient: dict, intervention: str = None,
                      advance_seconds: float = 10, **intervention_kwargs) -> dict:
    """
    Full Pulse before/after bridge for one patient.

    Parameters
    ----------
    raw_patient : dict — same raw dict shape as NB9a's scoring functions
    intervention : str or None — 'exercise', 'smoking_cessation', or None (no-op advance)
    advance_seconds : float — confirmed pattern (test_pulse.py used 10s)

    Returns
    -------
    dict: {pulse_before, pulse_after, intervention_applied}
    """
    pulse = PulseEngine()
    try:
        patient_configuration = build_pulse_patient(raw_patient)
        success = initialize_engine_from_patient(pulse, patient_configuration, data_manager)
        if not success:
            raise RuntimeError('Pulse engine failed to initialize from patient configuration.')

        print('✅ Patient initialized from raw demographics')
        pulse_before = extract_state(pulse)
        print(f'  Baseline: {pulse_before}')

        if intervention is not None:
            apply_intervention(pulse, intervention, **intervention_kwargs)

        pulse.advance_time_s(advance_seconds)   # ← confirmed call, from test_pulse.py
        pulse_after = extract_state(pulse)
        print(f'  After {advance_seconds}s + {intervention or "no intervention"}: {pulse_after}')

        return {
            'pulse_before': pulse_before,
            'pulse_after': pulse_after,
            'intervention_applied': intervention,
            'advance_seconds': advance_seconds,
        }
    finally:
        pulse.clear()   # ← confirmed call, from test_pulse.py


print("✅ run_pulse_bridge() defined")


---
# Section 7 — Test

## What is being done
Runs the bridge on one hand-built patient. **This is expected to fail on the first real run** — that's not a sign this notebook is broken, it's the unverified sections (3 and 5) doing exactly what they're supposed to: failing loudly with a specific error pointing at which HowTo file to check, instead of silently producing wrong physiology numbers.

**When it fails:** read the printed error, open the HowTo file it names, find the real method/class name, and fix the corresponding line in Section 3 or 5 — the rest of the pipeline (data requests, advance, extract, clear) should keep working unchanged underneath whatever fix is needed there.


In [ ]:
test_patient = {
    'age': 45, 'gender': 'm', 'height': 178, 'weight': 82,
    'ap_hi': 130, 'ap_lo': 85,
}

print('=' * 60)
print('  SECTION 7: Pulse bridge test')
print('=' * 60)
try:
    result = run_pulse_bridge(test_patient, intervention='exercise', intensity=0.6, advance_seconds=10)
    print('\n[SECTION 7 COMPLETE] ✅')
    print(result)
except Exception as e:
    print(f'\n❌ Failed at: {type(e).__name__}: {e}')
    print('   This is expected if Sections 3/5 haven\'t been corrected against')
    print('   HowTo_EngineUse.py / HowTo_Exercise.py yet — see header table for')
    print('   exactly which piece to check based on where this failed.')
